# MediaWiki Action API Tutorial for Social Scientists

This notebook introduces the MediaWiki Action API, which allows you to
programmatically access and retrieve data from Wikipedia and other
MediaWiki-powered sites. This is particularly useful for social science research
involving:

- Content analysis of Wikipedia articles
- Tracking edits and controversies
- Analyzing information dissemination
- Studying online collaboration

## What is the MediaWiki Action API?

The MediaWiki Action API is a web service that provides access to wiki features
like authentication, page operations, and search. It's RESTful and returns data
in JSON format, making it easy to work with in Python.


## Best Practices and Tips

### 1. Rate Limiting

Always be respectful of Wikipedia's servers. Add delays between requests:

```python
time.sleep(0.5)  # Wait 0.5 seconds between requests
```

### 2. Error Handling

Add try-except blocks for robust code:

```python
try:
    response = requests.get(API_URL, params=params)
    response.raise_for_status()
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")
```

### 3. User Agent

Identify your application (especially for research):

```python
headers = {
    'User-Agent': 'YourProjectName/1.0 (your.email@example.com)'
}
response = requests.get(API_URL, params=params, headers=headers)
```

### 4. API Documentation

Full documentation: https://www.mediawiki.org/wiki/API:Main_page


## Additional Resources

- [MediaWiki API Documentation](https://www.mediawiki.org/wiki/API:Main_page)
- [Wikipedia API Sandbox](https://en.wikipedia.org/wiki/Special:ApiSandbox) -
  Test queries interactively
- [PyWikibot](https://www.mediawiki.org/wiki/Manual:Pywikibot) - Python library
  for more advanced Wikipedia interaction
- [PAWS](https://www.mediawiki.org/wiki/PAWS) - Platform for running Jupyter
  notebooks with Wikipedia data


In [ ]:
import requests
import json
import pandas as pd
from datetime import datetime
import time

# Define the API endpoint and polite defaults
API_URL = "https://en.wikipedia.org/w/api.php"
USER_AGENT = "MediaWikiAPITutorial/1.0 (adashiyj@gmail.com)"
REQUEST_DELAY = 0.5  # seconds between requests
TIMEOUT = 30

session = requests.Session() # session persists certain parameters across multiple requests (more efficient than making individual requests)
session.headers.update({"User-Agent": USER_AGENT}) # this tells Wikipedia who is making the requests

def api_get(params: dict) -> dict | None: # Returns either a dictionary or None
    """Make a safe request to the MediaWiki API with retries and rate limiting."""
    try:
        response = session.get(API_URL, params=params, timeout=TIMEOUT)
        response.raise_for_status() # checks if the HTTP request was successful 
        data = response.json()
    except requests.exceptions.RequestException as e: 
        print(f"Request error: {e}")
        return None
    except json.JSONDecodeError: 
        print("Response was not valid JSON")
        return None
    finally: # this block always executes, whether an exception occurred or not
        time.sleep(REQUEST_DELAY)

    return data

## Use Case 1: Retrieving Article Content

Let's start by retrieving the content of a Wikipedia article. This is useful
when you want to analyze the text of specific articles.


In [ ]:
def get_article_content(title):
    """
    Retrieve the full text content of a Wikipedia article.
    
    Parameters:
    title (str): The title of the Wikipedia article
    
    Returns:
    str: The article content in plain text
    """
    params = {
        'action': 'query',
        'format': 'json',
        'titles': title,
        'prop': 'extracts',
        'explaintext': True,  # Get plain text instead of HTML
        'exsectionformat': 'plain'
    }
    
    data = api_get(params)
    if not data or 'query' not in data:
        return "Request failed or no data"
    
    # Extract the page content
    pages = data['query']['pages']
    page_id = list(pages.keys())[0]
    
    if page_id == '-1':
        return "Article not found"
    
    return pages[page_id].get('extract', 'No content available')

# Example: Get content about "Climate change"
article_title = "Social robot"
content = get_article_content(article_title)

# Display first 500 characters
print(f"Article: {article_title}")
print(f"\nFirst 500 characters:\n{content[:500]}...")
print(f"\nTotal length: {len(content)} characters")

## Use Case 2: Getting Article Metadata

Often you need metadata about articles, such as when they were last modified,
how many views they get, or their categories.


In [ ]:
def get_article_metadata(title):
    """
    Retrieve metadata about a Wikipedia article.
    
    Parameters:
    title (str): The title of the Wikipedia article
    
    Returns:
    dict: Dictionary containing article metadata
    """
    params = {
        'action': 'query',
        'format': 'json',
        'titles': title,
        'prop': 'info|categories',
        'inprop': 'url|displaytitle'
    }
    
    data = api_get(params)
    if not data or 'query' not in data:
        return {}
    
    pages = data['query']['pages']
    page_id = list(pages.keys())[0]
    page = pages[page_id]
    
    metadata = {
        'title': page.get('title', 'N/A'),
        'page_id': page_id,
        'url': page.get('fullurl', 'N/A'),
        'last_modified': page.get('touched', 'N/A'),
        'length': page.get('length', 'N/A'),
        'categories': [cat['title'] for cat in page.get('categories', [])]
    }
    
    return metadata

# Example usage
metadata = get_article_metadata("Social robot")
print("Article Metadata:")
print(f"Title: {metadata.get('title', 'N/A')}")
print(f"Page ID: {metadata.get('page_id', 'N/A')}")
print(f"URL: {metadata.get('url', 'N/A')}")
print(f"Last Modified: {metadata.get('last_modified', 'N/A')}")
print(f"Length: {metadata.get('length', 'N/A')} bytes")
print(f"\nFirst 5 categories:")
for cat in metadata.get('categories', [])[:5]:
    print(f"  - {cat}")

## Use Case 3: Searching Wikipedia

The search functionality is useful when you want to find articles related to
specific topics or keywords.


In [ ]:
def search_wikipedia(query, limit=10):
    """
    Search Wikipedia for articles matching a query.
    
    Parameters:
    query (str): Search term
    limit (int): Maximum number of results to return
    
    Returns:
    list: List of dictionaries containing search results
    """
    params = {
        'action': 'query',
        'format': 'json',
        'list': 'search',
        'srsearch': query,
        'srlimit': limit
    }
    
    data = api_get(params)
    if not data or 'query' not in data:
        return []
    
    results = []
    for item in data['query'].get('search', []):
        results.append({
            'title': item['title'],
            'snippet': item['snippet'],
            'wordcount': item['wordcount'],
            'timestamp': item['timestamp']
        })
    
    return results

# Example: Search for articles about misinformation
search_results = search_wikipedia("human-robot interaction", limit=5)

print("Search Results for 'human-robot interaction':\n")
for i, result in enumerate(search_results, 1):
    print(f"{i}. {result['title']}")
    print(f"   Word count: {result['wordcount']}")
    print(f"   Last updated: {result['timestamp']}")
    print()

## Use Case 4: Analyzing Edit History

For research on Wikipedia dynamics, controversies, or collaboration, examining
edit history is crucial.


In [ ]:
def get_revision_history(title, limit=50):
    """
    Retrieve the revision history of a Wikipedia article.
    
    Parameters:
    title (str): The title of the Wikipedia article
    limit (int): Number of revisions to retrieve
    
    Returns:
    pd.DataFrame: DataFrame containing revision information
    """
    params = {
        'action': 'query',
        'format': 'json',
        'titles': title,
        'prop': 'revisions',
        'rvprop': 'timestamp|user|size|comment',
        'rvlimit': limit
    }
    
    data = api_get(params)
    if not data or 'query' not in data:
        return pd.DataFrame([])
    
    pages = data['query']['pages']
    page_id = list(pages.keys())[0]
    revisions = pages[page_id].get('revisions', [])
    
    # Convert to DataFrame for easier analysis
    df = pd.DataFrame(revisions)
    if not df.empty:
        df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    return df

# Example: Analyze edit history of a controversial topic
revisions_df = get_revision_history("ICE", limit=20)

print("Recent Revisions of 'ICE':\n")
if not revisions_df.empty:
    print(revisions_df[['timestamp', 'user', 'size']].head(10))

# Basic statistics
print(f"\nTotal revisions retrieved: {len(revisions_df)}")
if not revisions_df.empty:
    print(f"Unique editors: {revisions_df['user'].nunique()}")
    print(f"Average article size: {revisions_df['size'].mean():.0f} bytes")

## Use Case 5: Getting Links and References

Analyzing how articles link to each other can reveal information networks and
topic relationships.


In [ ]:
def get_article_links(title, limit=50):
    """
    Get all links from a Wikipedia article to other articles.
    
    Parameters:
    title (str): The title of the Wikipedia article
    limit (int): Maximum number of links to retrieve
    
    Returns:
    list: List of linked article titles
    """
    params = {
        'action': 'query',
        'format': 'json',
        'titles': title,
        'prop': 'links',
        'pllimit': limit,
        'plnamespace': 0  # Only main namespace (articles)
    }
    
    data = api_get(params)
    if not data or 'query' not in data:
        return []
    
    pages = data['query']['pages']
    page_id = list(pages.keys())[0]
    links = pages[page_id].get('links', [])
    
    return [link['title'] for link in links]

# Example: Get links from an article
links = get_article_links("Social robot", limit=30)

print("Articles linked from 'Social robot':\n")
for i, link in enumerate(links[:20], 1):
    print(f"{i}. {link}")

print(f"\nTotal links retrieved: {len(links)}")

## Use Case 6: Comparing Multiple Articles

Let's create a practical example that compares multiple articles on related
topics.


In [ ]:
def compare_articles(titles):
    """
    Compare multiple Wikipedia articles.
    
    Parameters:
    titles (list): List of article titles to compare
    
    Returns:
    pd.DataFrame: Comparison table
    """
    comparison_data = []
    
    for title in titles:
        # Get metadata
        metadata = get_article_metadata(title)
        
        # Get content length
        content = get_article_content(title)
        
        # Get number of links
        links = get_article_links(title, limit=500)
        
        comparison_data.append({
            'Title': title,
            'Article Length (chars)': len(content),
            'Size (bytes)': metadata.get('length', 'N/A'),
            'Outgoing Links': len(links),
            'Categories': len(metadata.get('categories', [])),
            'Last Modified': str(metadata.get('last_modified', ''))[:10]
        })
    
    return pd.DataFrame(comparison_data)

# Example: Compare social robots articles
robots = ["Furhat", "NAO", "Pepper", "Reachy Mini", "Paro", "Jibo"]
comparison_df = compare_articles(robots)

print("Comparison of Social Robots Articles:\n")
print(comparison_df.to_string(index=False))

## Use Case 7: Pageviews (REST API)

The Wikimedia REST API provides pageview time‑series for articles. This is
useful when you want to compare article popularity over a specific period.

Docs: https://wikimedia.org/api/rest_v1/


In [ ]:
def get_pageviews(article, project="en.wikipedia", start="20260101", end="20260131", access="all-access", agent="user", granularity="daily"):
    """
    Get pageviews for a Wikipedia article using the Wikimedia REST API.

    Parameters:
    article (str): Article title (spaces allowed)
    project (str): e.g., 'en.wikipedia'
    start (str): YYYYMMDD
    end (str): YYYYMMDD
    access (str): all-access | desktop | mobile-app | mobile-web
    agent (str): user | spider | bot | all-agents
    granularity (str): daily | monthly

    Returns:
    pd.DataFrame: Date and views
    """
    base = "https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article"
    article = article.replace(" ", "_")
    url = f"{base}/{project}/{access}/{agent}/{article}/{granularity}/{start}/{end}"

    try:
        response = session.get(url, timeout=TIMEOUT)
        response.raise_for_status()
        data = response.json()
    except requests.exceptions.RequestException as e:
        print(f"Request error: {e}")
        return pd.DataFrame([])
    except json.JSONDecodeError:
        print("Response was not valid JSON")
        return pd.DataFrame([])
    finally:
        time.sleep(REQUEST_DELAY)

    items = data.get("items", [])
    if not items:
        return pd.DataFrame([])

    df = pd.DataFrame(items)
    df["date"] = pd.to_datetime(df["timestamp"].str.slice(0, 8))
    return df[["date", "views"]]

# Example: Pageviews for "Social robot" in Jan 2024
pv = get_pageviews("Social robot", start="20260101", end="20260131")
print(pv.head())

## Student Exercises (submission)

Complete **Exercise 1 and Exercise 2** below. **Exercise 3 is optional (bonus)**
if you want a challenge.

- Do the work **in this notebook** (recommended) or in a new notebook named
  `mediawiki_api_exercises.ipynb`.
- If you use a new notebook, copy the helper functions above or import them, and
  include this tutorial notebook in your GitHub repo as well.
- Save and push your notebook(s) to GitHub for submission.

Each exercise includes a **TODO** section where you add your code and a short
written reflection.


### Exercise 0 — Pageviews (REST API)

**Goal:** Collect pageview data for one article over a one‑month period.

**Deliverables:**

- A table with `date` and `views` for your chosen article.
- A 3–5 sentence reflection on any trends you see.

**TODO:** Choose an article and a date range (one month).


In [ ]:
# TODO: Student work for Exercise 0 (pageviews)
def get_pageviews(article, project="en.wikipedia", start="20260201", end="20260228", access="all-access", agent="user", granularity="daily"):

    base = "https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article"
    article = article.replace(" ", "_")
    url = f"{base}/{project}/{access}/{agent}/{article}/{granularity}/{start}/{end}"

    try:
        response = session.get(url, timeout=TIMEOUT)
        response.raise_for_status()
        data = response.json()
    except requests.exceptions.RequestException as e:
        print(f"Request error: {e}")
        return pd.DataFrame([])
    except json.JSONDecodeError:
        print("Response was not valid JSON")
        return pd.DataFrame([])
    finally:
        time.sleep(REQUEST_DELAY)

    items = data.get("items", [])
    if not items:
        return pd.DataFrame([])

    df = pd.DataFrame(items)
    df["date"] = pd.to_datetime(df["timestamp"].str.slice(0, 8))
    return df[["date", "views"]]

# Example: Pageviews for "67th Annual Grammy Awards" in Feb 2026
pv = get_pageviews("67th Annual Grammy Awards", start="20260201", end="20260228")
pv

**Reflection:**
The 67th Annual Grammy Awards took place on February 2, 2026. Pageviews for "Grammy" on Wikipedia reached the peak on the day of the award, which makes sense since everyone wanted to know about the nominees and winners. After that, views dropped off pretty quickly as people moved on to other things.

### Tips for keeping it simple

If you want a low‑stress path:

- Choose familiar topics (e.g., sports teams, public figures, films, or cities).
- Use 5 articles (not 8) for Exercise 1.
- For Exercise 2, start with 15–20 revisions instead of 30.
- For Exercise 3 (optional), only collect links from the seed article (skip the
  second hop).


### Exercise 1 — Build a mini dataset

**Goal:** Collect structured data from 5–8 Wikipedia articles related to a topic
of your choice.

**Deliverables:**

- A pandas DataFrame with at least these columns: `title`, `length_chars`,
  `num_categories`, `num_links`, `last_modified`.
- Save the DataFrame to `data/processed/exercise1_wikipedia_dataset.csv`.
- A 3–5 sentence reflection on what you noticed in the data.

**TODO:** Choose a topic and list 5–8 article titles you will analyze.


In [ ]:
# Step 1: Choose a topic and list some relevant article titles
def search_wikipedia(query, limit=10):

    params = {
        'action': 'query',
        'format': 'json',
        'list': 'search',
        'srsearch': query,
        'srlimit': limit
    }
    
    data = api_get(params)
    if not data or 'query' not in data:
        return []
    
    results = []
    for item in data['query'].get('search', []):
        results.append({
            'title': item['title'],
        })
    
    return results

# Search for articles
search_results = search_wikipedia("Most overrated celebrities", limit=10)

print("Search Results for 'Most overrated celebrities':\n")
for i, result in enumerate(search_results, 1):
    print(f"{i}. {result['title']}")

In [ ]:
# Step 2: Select 5 articles, retrieve metadata, and save to a pandas DataFrame and CSV file
def compare_articles(titles):

    comparison_data = []
    
    for title in titles:
        # Get metadata
        metadata = get_article_metadata(title)
        
        # Get content length
        content = get_article_content(title)
        
        # Get number of links
        links = get_article_links(title, limit=500)
        
        comparison_data.append({
            'Title': title,
            'Article Length (chars)': len(content),
            'Categories': len(metadata.get('categories', [])),
            'Outgoing Links': len(links),
            'Last Modified': str(metadata.get('last_modified', ''))[:10]
        })
    
    return pd.DataFrame(comparison_data)

# Compare selected articles
articles = ["Paris Hilton", 
          "Anna Kournikova", 
          "Noel Gallagher", 
          "Bruce Springsteen",
          "Steve Perry"]

comparison_df = compare_articles(articles)
print("Comparison of Selected Articles:\n")
print(comparison_df.to_string(index=False))

# Save to a CSV file
comparison_df.to_csv('data/processed/exercise1_wikipedia_dataset.csv', index=False)

**Reflection (3–5 sentences):**

- What patterns did you notice in article length, categories, or links?
- Which article surprised you and why?


### Exercise 2 — Edit history analysis

**Goal:** Analyze edit activity for 1 controversial or rapidly changing topic.

**Deliverables:**

- A table of the most recent 30 revisions with `timestamp`, `user`, and `size`.
- A short summary with: total revisions retrieved, unique editors, and average
  size change.
- A 3–5 sentence reflection on what the edit history suggests about the topic.

**TODO:** Choose one article title to analyze.


In [ ]:
# TODO: Student work for Exercise 2


**Reflection (3–5 sentences):**

- What does the edit activity suggest about the topic’s stability or
  controversy?
- Do you notice patterns in editor activity or size changes?


### Exercise 3 (optional) — Link network snapshot

**Goal:** Build a small network of links starting from one seed article.

**Deliverables:**

- A list of the top 15 linked articles from your seed article.
- (Optional) A second list of links from **one** of those linked articles.
- A 3–5 sentence reflection on what the connections suggest about the topic
  area.

**TODO:** Choose a seed article.


In [ ]:
# TODO: Student work for Exercise 3 (optional)


**Reflection (3–5 sentences):**

- What clusters or themes appear in the linked articles?
- Does the network feel broad or tightly focused?

### Submission checklist

- [ ] Exercise 1 completed with CSV saved to `data/processed/`
- [ ] Exercise 2 completed with summary statistics
- [ ] Exercise 3 completed with two link lists
- [ ] Reflections written for each exercise
- [ ] Notebook(s) saved and pushed to GitHub
